# Twoje dane sprzedażowe: jakość, EDA i ważenie

Źródło: albim72/AI_analiza_danych_tools_11, DZIEN_2/sales_csv. Metoda profilowania inspirowana także pandas_experience_final.ipynb i kontrolą jakości z SWPS_2. 35 minut. Plik sales_original.csv zachowuje źródło bez zmian, sales_cwiczenie.csv jest celowo zmodyfikowaną kopią dydaktyczną.

Uruchom komórkę Start. Cały pakiet musi być rozpakowany. W Colab prześlij ZIP i rozpakuj go przez panel Pliki lub komendę `!unzip nazwa.zip`, a następnie ustaw katalog roboczy na folder pakietu.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'dane').exists() and (ROOT.parent/'dane').exists(): ROOT=ROOT.parent
if not (ROOT/'dane').exists(): raise FileNotFoundError('Rozpakuj cały pakiet i otwórz notebook w folderze zawierającym dane oraz narzedzia.')
sys.path.insert(0,str(ROOT/'narzedzia'))
from swps_tools import sales_quality, aggregate_chunks, to_cents, AmountCapper, make_transactions

## 1. Audyt i jednostka

Wczytaj dane/sales_cwiczenie.csv. Ustal, czy line_id opisuje zamówienie, pozycję czy klienta. Policz wiersze, unikalne klucze i flagi. Nie zakładaj, że oryginał z GitHuba zawierał te błędy: cztery kopie i cztery zmiany dodano na potrzeby tego warsztatu.

In [ ]:
raw=pd.read_csv(ROOT/'dane/sales_cwiczenie.csv',encoding='utf-8-sig')
flags=sales_quality(raw)
print('Wiersze:',len(raw),'unikalne pozycje:',raw.line_id.nunique())
print(flags.sum().to_string())
print('Wiersze z co najmniej jednym problemem:',flags.any(axis=1).sum())
assert len(raw)==724 and raw.line_id.nunique()==720
assert flags.any(axis=1).sum()==8

## 2. Kwarantanna i jawna reguła

Zachowaj oryginał. Oznacz kopie i niespójne rekordy. Zbuduj zbiór poprawnych pozycji, a pozostałe zapisz osobno. Nie naprawiaj kwoty tylko po to, by zgadzała się z wybranym wynikiem.

In [ ]:
clean=raw.loc[~flags.any(axis=1)].copy()
quarantine=raw.loc[flags.any(axis=1)].join(flags.add_prefix('flag_'))
clean['order_date']=pd.to_datetime(clean.order_date)
assert len(clean)==716 and clean.line_id.is_unique
print('Analiza:',len(clean),'kwarantanna:',len(quarantine))
print(quarantine[['line_id','order_date','discount','unit_price','net']].to_string(index=False))

## 3. EDA i agregacja

Oblicz kwotę netto i marżę pozycji. Porównaj średnią oraz medianę. Zrób ranking produktów i agregację miesięczną. Czy liczba line_id jest liczbą zamówień? Czy wolno połączyć sales i fuel tylko po numerze wiersza?

In [ ]:
clean['net_recalc']=clean.quantity*clean.unit_price*(1-clean.discount)
clean['margin']=clean.net_recalc-clean.quantity*clean.unit_cost
print(clean[['net_recalc','margin']].describe().to_string())
products=clean.groupby('product').agg(positions=('line_id','size'),net=('net_recalc','sum'),margin=('margin','sum')).sort_values('net',ascending=False)
monthly=clean.groupby(clean.order_date.dt.to_period('M'))[['net_recalc','margin']].sum()
print(products.to_string());print(monthly.to_string())
clean.net_recalc.hist(bins=30)
plt.xlabel('Wartość pozycji netto [PLN]');plt.ylabel('Liczba poprawnych pozycji');plt.title('Ćwiczenie sprzedażowe, n='+str(len(clean)));plt.show()
assert np.isclose(products.net.sum(),clean.net_recalc.sum())
# Brak order_id: nie ustalimy liczby zamówień. sales i fuel nie mają udokumentowanego wspólnego klucza.

## 4. Fuel: średnia wskaźników i wskaźnik łączny

Wczytaj fuel.csv. Wskaż rekord z zerowym dystansem. Dla dodatniego dystansu policz 100*litry/km dla każdej trasy, jego zwykłą średnią oraz 100*SUM(litry)/SUM(km). Wyjaśnij różnicę. Osobno pokaż wiersze z zerowym paliwem i wysokim wynikiem. Nie kasuj ich bez rozpoznania przyczyny.

In [ ]:
fuel=pd.read_csv(ROOT/'dane/fuel.csv',encoding='utf-8-sig')
print('Do wyjaśnienia, dystans <=0:',fuel.loc[fuel.distance_km.le(0)].to_string(index=False))
routes=fuel.loc[fuel.distance_km.gt(0)&fuel.fuel_liters.ge(0)].copy()
routes['l_per_100km']=100*routes.fuel_liters/routes.distance_km
unweighted=routes.l_per_100km.mean()
combined=100*routes.fuel_liters.sum()/routes.distance_km.sum()
weighted=np.average(routes.l_per_100km,weights=routes.distance_km)
print('Średnia tras:',unweighted,'L/100 km; wskaźnik łączny:',combined,'L/100 km')
print('Zero paliwa:',routes.loc[routes.fuel_liters.eq(0)].to_string(index=False))
print('Wysokie odczyty, do wyjaśnienia:',routes.loc[routes.l_per_100km.gt(20)].to_string(index=False))
assert np.isclose(weighted,combined)
assert len(routes)==119

## Raport
Podaj jednostkę, liczbę wykluczonych pozycji, wynik netto po jawnej regule oraz wyjaśnienie różnicy między średnią spalania tras i wskaźnikiem całej floty. Ujemna marża nie jest automatycznie błędem. Pliki nie mają wspólnego klucza, więc nie łączymy ich po indeksie.